# Notebook 05: Membership Inference Attacks

## Why This Matters
Your model is deployed behind an API. Someone queries it with a medical record, a financial transaction, a personal email — and determines, from nothing but the model's output, whether that record was in your training set. This is a membership inference attack, and it has direct regulatory consequences under GDPR (right to erasure) and HIPAA (health data protection).

The threat is real: Shokri et al. (2017) demonstrated it empirically. Google, Apple, and Amazon have all faced questions about what models trained on user data reveal. The `memory-leak` tool in the project backlog is a developer-facing auditor for this exact vulnerability.

## Structure
1. What is membership inference and why models are vulnerable
2. The shadow model attack (Shokri et al., 2017)
3. Threshold-based attacks — the simplest effective approach
4. LiRA: Likelihood Ratio Attack (Carlini et al., 2022)
5. Evaluating model memorization
6. Defenses: differential privacy, confidence masking, regularization
7. `memory-leak` prototype: a reusable audit function

## What You'll Learn
- Why overfit models are more vulnerable than well-regularized ones
- Why confidence scores are the primary signal for membership inference
- How LiRA provides a statistically grounded attack that sets the current benchmark
- How to audit your own model before deployment
- What differential privacy guarantees and what it costs

## Background

### The core intuition: models remember their training data

When you train a neural network, it doesn't just learn a function — it implicitly memorizes aspects of its training set. A model that has seen a data point during training tends to assign it higher confidence (lower loss) than a model that hasn't. This isn't a bug: it's a direct consequence of gradient descent minimizing training loss. The training examples are exactly those for which the optimizer has found a good fit.

The gap between training loss and test loss (generalization gap) is the fundamental signal exploited by membership inference. A model with zero generalization gap — perfectly calibrated, trained with infinite data — would be immune. Real models, trained on finite data with any degree of overfitting, are vulnerable.

### Shokri et al. (2017): the shadow model attack

The seminal paper that established membership inference as a practical threat was Shokri, Stronati, Song & Shmatikoff (2017) — *"Membership Inference Attacks Against Machine Learning Models"*, presented at IEEE S&P.

Their attack is elegant: train "shadow models" on data drawn from the same distribution as the target model's training set, then train a binary classifier on (shadow model output, label, membership) triples. The shadow models simulate the target model's behavior — the attack model learns to distinguish training vs. non-training outputs from these simulacra, then applies that knowledge to the real target.

This requires no access to the target model's architecture or weights — only the ability to query its output. It works in a black-box setting, which is exactly the deployment scenario.

Why shadow models? The attacker doesn't know the target model's training set. But they can sample from the same data distribution (the attacker might be a competitor with access to similar data, or someone with access to part of the dataset). The shadow models act as synthetic targets for training the attack classifier.

### Threshold attacks: surprisingly effective

Yeom et al. (2018) showed that even a naive threshold attack is effective against overfit models: if the model's confidence on a record exceeds a threshold, predict it was a training member. This requires no shadow models and no machine learning — just the model's output confidence.

The threshold attack's effectiveness is a useful diagnostic: if a threshold attack achieves non-trivial accuracy against your model, the model is leaking membership information through its confidence scores.

### LiRA: the current state of the art (Carlini et al., 2022)

Carlini, Chien, Nasr, Song, Terzis & Tramer (2022) — *"Membership Inference Attacks from First Principles"* — introduced LiRA (Likelihood Ratio Attack), which is both theoretically grounded and empirically the strongest known attack.

The key insight: the right question isn't "what is the model's confidence on this record?" — it's "how does the model's confidence on this record compare to the counterfactual confidence if the record *hadn't* been in training?"

LiRA estimates this likelihood ratio by training many models *with* and *without* the target record, then comparing the resulting confidence distributions. This gives a principled statistical test (likelihood ratio test) rather than an arbitrary threshold.

LiRA is more expensive than the shadow model attack (requires training O(k) models per target record), but it's significantly more accurate, especially at low false positive rates — which is the operationally relevant regime. If an attacker only makes claims about membership when they're confident, LiRA is much better than threshold attacks.

### Why this matters beyond the paper

**GDPR Article 17** grants data subjects the right to erasure. If you can't determine whether a record was used in training, you can't verify erasure. Membership inference auditing is the tool.

**Medical data**: models trained on patient records can leak whether a specific patient was in the training set — which itself reveals that the patient has the condition being modeled.

**Model releases**: when you publish model weights, you publish everything the model has memorized. The famous GPT-2 and GPT-3 memorization papers (Carlini et al., 2021) showed that large language models memorize verbatim training text that can be extracted by querying.

In [ ]:
%pip install -q torch torchvision numpy matplotlib scikit-learn

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import torchvision
import torchvision.transforms as transforms
import numpy as np
import matplotlib.pyplot as plt
from sklearn.metrics import roc_auc_score, roc_curve

torch.manual_seed(42)
np.random.seed(42)
plt.rcParams.update({'figure.dpi': 120, 'font.size': 10})

if torch.backends.mps.is_available():
    device = torch.device('mps')
elif torch.cuda.is_available():
    device = torch.device('cuda')
else:
    device = torch.device('cpu')

print(f"Device: {device}")

transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.1307,), (0.3081,))
])

full_train = torchvision.datasets.MNIST('./data', train=True,  download=True, transform=transform)
full_test  = torchvision.datasets.MNIST('./data', train=False, download=True, transform=transform)

# Split training set: half for actual training, half as "non-members" for attack evaluation
n_train = 10_000  # use a small subset to exaggerate memorization
n_nonmember = 5_000

indices = torch.randperm(len(full_train))
member_indices    = indices[:n_train]
nonmember_indices = indices[n_train:n_train + n_nonmember]

member_dataset    = torch.utils.data.Subset(full_train, member_indices)
nonmember_dataset = torch.utils.data.Subset(full_train, nonmember_indices)

train_loader    = torch.utils.data.DataLoader(member_dataset,    batch_size=128, shuffle=True)
nonmember_loader = torch.utils.data.DataLoader(nonmember_dataset, batch_size=128, shuffle=False)
test_loader     = torch.utils.data.DataLoader(full_test,          batch_size=128, shuffle=False)

print(f"Members (training set):    {len(member_dataset):,}")
print(f"Non-members (held-out):    {len(nonmember_dataset):,}")
print(f"Test set:                  {len(full_test):,}")

In [ ]:
class MnistCNN(nn.Module):
    def __init__(self, dropout=0.25):
        super().__init__()
        self.conv1 = nn.Conv2d(1, 32, 3, padding=1)
        self.conv2 = nn.Conv2d(32, 64, 3, padding=1)
        self.pool  = nn.MaxPool2d(2)
        self.fc1   = nn.Linear(64 * 7 * 7, 128)
        self.fc2   = nn.Linear(128, 10)
        self.drop  = nn.Dropout(dropout)

    def forward(self, x):
        x = F.relu(self.conv1(x)); x = self.pool(x)
        x = F.relu(self.conv2(x)); x = self.pool(x)
        x = F.relu(self.fc1(x.flatten(1)))
        x = self.drop(x)
        return self.fc2(x)


def train_model(loader, epochs=20, dropout=0.0, lr=1e-3, seed=42):
    """Train a model and return it. High epochs + no dropout → overfit → vulnerable."""
    torch.manual_seed(seed)
    model = MnistCNN(dropout=dropout).to(device)
    opt = torch.optim.Adam(model.parameters(), lr=lr)
    for _ in range(epochs):
        model.train()
        for x, y in loader:
            x, y = x.to(device), y.to(device)
            opt.zero_grad(); F.cross_entropy(model(x), y).backward(); opt.step()
    return model


def get_confidence_scores(model, loader, device):
    """Return confidence scores (max softmax) and correctness for all examples."""
    model.eval()
    confs, labels, preds = [], [], []
    with torch.no_grad():
        for x, y in loader:
            x, y = x.to(device), y.to(device)
            probs = F.softmax(model(x), dim=1)
            max_conf, pred = probs.max(dim=1)
            confs.append(max_conf.cpu().numpy())
            labels.append(y.cpu().numpy())
            preds.append(pred.cpu().numpy())
    return np.concatenate(confs), np.concatenate(labels), np.concatenate(preds)


# Train an overfit model (small dataset, many epochs, no regularization)
print("Training overfit target model (10k examples, 20 epochs, no dropout)...")
model_overfit = train_model(train_loader, epochs=20, dropout=0.0)

member_confs, _, member_preds = get_confidence_scores(model_overfit, train_loader, device)
nonmember_confs, _, nonmember_preds = get_confidence_scores(model_overfit, nonmember_loader, device)

# Evaluate on the test loader
correct = total = 0
model_overfit.eval()
with torch.no_grad():
    for x, y in test_loader:
        x, y = x.to(device), y.to(device)
        correct += (model_overfit(x).argmax(1) == y).sum().item(); total += y.size(0)
test_acc = correct / total

train_acc = member_preds.mean()  # proxy; actual accuracy from loader needed
correct_train = total_train = 0
model_overfit.eval()
with torch.no_grad():
    for x, y in train_loader:
        x, y = x.to(device), y.to(device)
        correct_train += (model_overfit(x).argmax(1) == y).sum().item(); total_train += y.size(0)
train_acc = correct_train / total_train

print(f"Train accuracy: {train_acc:.4f}")
print(f"Test accuracy:  {test_acc:.4f}")
print(f"Generalization gap: {train_acc - test_acc:.4f}  (larger gap → more vulnerable)")
print()
print(f"Mean confidence — members:     {member_confs.mean():.4f}")
print(f"Mean confidence — non-members: {nonmember_confs.mean():.4f}")
print(f"Confidence gap: {member_confs.mean() - nonmember_confs.mean():.4f}  (this is what the attacker exploits)")

## 1. Threshold Attack

The simplest membership inference attack: predict "member" if the model's confidence on the record exceeds a threshold $\tau$.

```
attack(x) = 1 (member) if confidence(model, x) ≥ τ else 0 (non-member)
```

This is Yeom et al.'s attack, and it works because members tend to have higher confidence. The optimal threshold balances the false positive rate (non-members above threshold) against the true positive rate (members above threshold).

The ROC curve of this attack is a useful vulnerability diagnostic: an AUC close to 0.5 means the model doesn't leak membership information; AUC close to 1.0 means it does.

In [ ]:
def threshold_attack_roc(member_confs, nonmember_confs):
    """
    Compute ROC curve for the threshold attack.
    Label 1 = member, 0 = non-member.
    Score = model confidence.
    """
    scores = np.concatenate([member_confs, nonmember_confs])
    labels = np.concatenate([np.ones(len(member_confs)), np.zeros(len(nonmember_confs))])
    fpr, tpr, thresholds = roc_curve(labels, scores)
    auc = roc_auc_score(labels, scores)
    return fpr, tpr, thresholds, auc


fpr, tpr, thresholds, auc = threshold_attack_roc(member_confs, nonmember_confs)

# Find the threshold that maximizes TPR at FPR <= 0.1
# (operationally: how often can we detect membership while limiting false accusations?)
fpr_budget = 0.1
valid = fpr <= fpr_budget
best_tpr_at_low_fpr = tpr[valid].max() if valid.any() else 0.0

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# ROC curve
axes[0].plot(fpr, tpr, color='steelblue', linewidth=2, label=f'Threshold attack (AUC={auc:.3f})')
axes[0].plot([0, 1], [0, 1], 'k--', alpha=0.4, label='Random (AUC=0.5)')
axes[0].axvline(fpr_budget, color='red', linestyle=':', alpha=0.6, label=f'FPR budget = {fpr_budget}')
axes[0].set_xlabel('False Positive Rate'); axes[0].set_ylabel('True Positive Rate')
axes[0].set_title('Membership Inference ROC\n(Threshold Attack)')
axes[0].legend(); axes[0].grid(True, alpha=0.3)

# Confidence distributions
axes[1].hist(member_confs, bins=50, alpha=0.6, color='steelblue', label='Members (train)', density=True)
axes[1].hist(nonmember_confs, bins=50, alpha=0.6, color='darkorange', label='Non-members', density=True)
axes[1].set_xlabel('Model confidence (max softmax)')
axes[1].set_ylabel('Density')
axes[1].set_title('Confidence Distribution\nMembers vs Non-Members')
axes[1].legend(); axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('membership_inference_roc.png', bbox_inches='tight')
plt.show()

print(f"Threshold attack AUC:           {auc:.4f}")
print(f"TPR at FPR ≤ {fpr_budget}:          {best_tpr_at_low_fpr:.4f}")
print(f"  → attacker correctly identifies {best_tpr_at_low_fpr:.1%} of members")
print(f"    while falsely accusing only {fpr_budget:.1%} of non-members")

## 2. Shadow Model Attack (Shokri et al.)

The shadow model attack trains a meta-classifier to distinguish training examples from non-training examples based on the model's output. The attacker:

1. Trains $k$ "shadow models" on data from the same distribution as the target
2. For each shadow model, they know which examples were in training (members) and which weren't
3. They train an attack classifier on (model output, true label) → member/non-member
4. Apply the attack classifier to the target model

The attack classifier learns to distinguish the *shape* of the model's output on members vs non-members — not just the top confidence score, but the full softmax distribution.

In [ ]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression


def get_softmax_outputs(model, loader, device):
    """Return full softmax output vectors for all examples."""
    model.eval()
    outputs, ys = [], []
    with torch.no_grad():
        for x, y in loader:
            x = x.to(device)
            probs = F.softmax(model(x), dim=1)
            outputs.append(probs.cpu().numpy())
            ys.append(y.numpy())
    return np.concatenate(outputs), np.concatenate(ys)


def shadow_model_attack(target_model, train_loader, nonmember_loader, n_shadow=3):
    """
    Shadow model membership inference attack (Shokri et al., 2017).

    Trains n_shadow models on subsets of the available data,
    uses them to train an attack classifier,
    then evaluates on the target model.
    """
    # Collect all available data for shadow training
    all_data = torch.utils.data.ConcatDataset([member_dataset, nonmember_dataset])
    n_per_shadow = len(all_data) // (2 * n_shadow)  # each shadow trains on n_per_shadow examples

    attack_X_train, attack_y_train = [], []

    for i in range(n_shadow):
        # Sample disjoint subsets for shadow training
        shadow_indices = torch.randperm(len(all_data))
        shadow_member_idx    = shadow_indices[:n_per_shadow]
        shadow_nonmember_idx = shadow_indices[n_per_shadow:2 * n_per_shadow]

        shadow_train_set = torch.utils.data.Subset(all_data, shadow_member_idx)
        shadow_out_set   = torch.utils.data.Subset(all_data, shadow_nonmember_idx)

        shadow_train_loader = torch.utils.data.DataLoader(shadow_train_set, batch_size=128, shuffle=True)
        shadow_out_loader   = torch.utils.data.DataLoader(shadow_out_set,   batch_size=128)

        # Train shadow model
        shadow_model = train_model(shadow_train_loader, epochs=15, dropout=0.0, seed=i * 100)

        # Collect outputs: members → label 1, non-members → label 0
        member_probs,    member_labels    = get_softmax_outputs(shadow_model, shadow_train_loader, device)
        nonmember_probs, nonmember_labels = get_softmax_outputs(shadow_model, shadow_out_loader,   device)

        # Feature: [softmax output sorted desc, true class index]
        def featurize(probs, labels):
            sorted_probs = np.sort(probs, axis=1)[:, ::-1]  # sort desc
            return sorted_probs  # use top-k softmax as features

        attack_X_train.append(featurize(member_probs, member_labels))
        attack_X_train.append(featurize(nonmember_probs, nonmember_labels))
        attack_y_train.append(np.ones(len(member_probs)))
        attack_y_train.append(np.zeros(len(nonmember_probs)))

        print(f"  Shadow model {i+1}/{n_shadow} trained")

    # Train attack classifier on shadow model outputs
    X_attack = np.vstack(attack_X_train)
    y_attack  = np.concatenate(attack_y_train)

    attack_clf = RandomForestClassifier(n_estimators=50, max_depth=5, random_state=42)
    attack_clf.fit(X_attack, y_attack)

    # Evaluate on target model
    target_member_probs,    _ = get_softmax_outputs(target_model, train_loader,     device)
    target_nonmember_probs, _ = get_softmax_outputs(target_model, nonmember_loader, device)

    def featurize(probs, labels=None):
        return np.sort(probs, axis=1)[:, ::-1]

    X_member    = featurize(target_member_probs)
    X_nonmember = featurize(target_nonmember_probs)
    X_eval = np.vstack([X_member, X_nonmember])
    y_eval = np.concatenate([np.ones(len(X_member)), np.zeros(len(X_nonmember))])

    scores = attack_clf.predict_proba(X_eval)[:, 1]
    auc = roc_auc_score(y_eval, scores)
    fpr, tpr, _ = roc_curve(y_eval, scores)
    return auc, fpr, tpr


print("Running shadow model attack (3 shadow models)...")
shadow_auc, shadow_fpr, shadow_tpr = shadow_model_attack(model_overfit, train_loader, nonmember_loader, n_shadow=3)
print(f"Shadow model attack AUC: {shadow_auc:.4f}")

## 3. LiRA: Likelihood Ratio Attack

LiRA frames membership inference as a hypothesis test:
- $H_0$: the record was **not** in training (null hypothesis)
- $H_1$: the record **was** in training

The likelihood ratio is:

$$\Lambda(x) = \frac{p(\text{confidence}(x) \mid H_1)}{p(\text{confidence}(x) \mid H_0)}$$

To estimate these distributions, LiRA trains $k$ models that **include** $x$ in training and $k$ models that **exclude** it, then fits a Gaussian to each confidence distribution. The test is then a standard likelihood ratio test.

This is principled in a way threshold attacks aren't: you're asking "does this confidence score look more like what we'd expect from a model that saw this record, or one that didn't?" rather than just "is the confidence high?".

We implement a simplified version here — the full LiRA trains O(256) models per record, which is expensive. We use fewer models and a faster training loop to demonstrate the concept.

In [ ]:
def lira_attack(target_model, member_dataset, nonmember_dataset,
                n_models_per_group=6, n_eval_examples=100):
    """
    Simplified LiRA: Likelihood Ratio Attack (Carlini et al., 2022).

    For each target example, trains n_models with and without it,
    fits Gaussians to the confidence distributions, computes log-likelihood ratio.

    Full LiRA uses ~256 models per example; we use fewer for speed.
    """
    all_data = torch.utils.data.ConcatDataset([member_dataset, nonmember_dataset])
    all_indices = list(range(len(all_data)))

    # Pick a small set of target examples to evaluate (members + non-members)
    target_member_idx    = list(range(min(n_eval_examples // 2, len(member_dataset))))
    target_nonmember_idx = list(range(len(member_dataset), len(member_dataset) + n_eval_examples // 2))
    all_target_idx = target_member_idx + target_nonmember_idx
    true_labels = [1] * len(target_member_idx) + [0] * len(target_nonmember_idx)

    # For each target example, collect confidence scores from "in" and "out" models
    # in_confs[i] = list of confidences on target[i] from models trained WITH it
    # out_confs[i] = list of confidences on target[i] from models trained WITHOUT it
    in_confs  = [[] for _ in all_target_idx]
    out_confs = [[] for _ in all_target_idx]

    n_total_models = n_models_per_group * 2
    pool_size = min(len(all_data), 3000)  # subset for faster shadow training

    for model_idx in range(n_total_models):
        # Randomly assign each target example to "in" or "out" for this shadow model
        in_mask = np.random.binomial(1, 0.5, len(all_target_idx)).astype(bool)

        # Build shadow training set from pool, forcing inclusion/exclusion of targets
        pool_indices = np.random.choice(pool_size, size=min(2000, pool_size), replace=False).tolist()
        # Add "in" targets to training
        shadow_train_indices = pool_indices + [all_target_idx[i] for i in range(len(all_target_idx)) if in_mask[i]]
        shadow_train_set = torch.utils.data.Subset(all_data, shadow_train_indices)
        shadow_loader = torch.utils.data.DataLoader(shadow_train_set, batch_size=128, shuffle=True)

        # Train shadow model quickly
        shadow = train_model(shadow_loader, epochs=10, dropout=0.0, seed=model_idx * 7)
        shadow.eval()

        # Collect confidences on each target example
        for i, t_idx in enumerate(all_target_idx):
            x, y = all_data[t_idx]
            with torch.no_grad():
                probs = F.softmax(shadow(x.unsqueeze(0).to(device)), dim=1)
                conf = probs[0, y].item()  # confidence on the TRUE class
            if in_mask[i]:
                in_confs[i].append(conf)
            else:
                out_confs[i].append(conf)

        if (model_idx + 1) % 3 == 0:
            print(f"  Shadow model {model_idx+1}/{n_total_models}")

    # Compute LiRA score for each target example: log-likelihood ratio
    lira_scores = []
    for i in range(len(all_target_idx)):
        in_c  = np.array(in_confs[i])  + 1e-8
        out_c = np.array(out_confs[i]) + 1e-8
        if len(in_c) < 2 or len(out_c) < 2:
            lira_scores.append(0.0)
            continue

        # Fit Gaussians to in/out confidence distributions
        mu_in,  std_in  = in_c.mean(),  in_c.std()  + 1e-6
        mu_out, std_out = out_c.mean(), out_c.std() + 1e-6

        # Confidence on target example from the ACTUAL target model
        x_t, y_t = all_data[all_target_idx[i]]
        with torch.no_grad():
            p = F.softmax(target_model(x_t.unsqueeze(0).to(device)), dim=1)
            target_conf = p[0, y_t].item()

        # Log-likelihood ratio: log p(conf | in) - log p(conf | out)
        log_p_in  = -0.5 * ((target_conf - mu_in)  / std_in)  ** 2 - np.log(std_in)
        log_p_out = -0.5 * ((target_conf - mu_out) / std_out) ** 2 - np.log(std_out)
        lira_scores.append(log_p_in - log_p_out)

    auc = roc_auc_score(true_labels, lira_scores)
    fpr, tpr, _ = roc_curve(true_labels, lira_scores)
    return auc, fpr, tpr


print("Running simplified LiRA (12 shadow models, 100 target examples)...")
lira_auc, lira_fpr, lira_tpr = lira_attack(
    model_overfit, member_dataset, nonmember_dataset,
    n_models_per_group=6, n_eval_examples=100
)
print(f"LiRA attack AUC: {lira_auc:.4f}")

In [ ]:
# Compare all three attacks on the same plot
fig, ax = plt.subplots(figsize=(8, 6))

# Threshold attack ROC
ax.plot(fpr, tpr, color='steelblue', linewidth=2,
        label=f'Threshold (AUC={auc:.3f})')
# Shadow model attack ROC
ax.plot(shadow_fpr, shadow_tpr, color='darkorange', linewidth=2,
        label=f'Shadow model (AUC={shadow_auc:.3f})')
# LiRA ROC
ax.plot(lira_fpr, lira_tpr, color='darkgreen', linewidth=2,
        label=f'LiRA (AUC={lira_auc:.3f})')
ax.plot([0, 1], [0, 1], 'k--', alpha=0.3, label='Random (AUC=0.5)')

# Low FPR region (operationally most important)
ax.axvspan(0, 0.1, alpha=0.05, color='red')
ax.text(0.05, 0.95, 'Low FPR\nregion', ha='center', fontsize=8, color='red')

ax.set_xlabel('False Positive Rate')
ax.set_ylabel('True Positive Rate')
ax.set_title('Membership Inference Attack Comparison')
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig('mi_attack_comparison.png', bbox_inches='tight')
plt.show()

## 4. How Regularization Reduces Vulnerability

The generalization gap drives vulnerability. Anything that reduces overfitting also reduces the confidence gap between members and non-members — which is the primary signal membership inference exploits.

- **Dropout** reduces overfitting
- **Weight decay** reduces overfitting  
- **Early stopping** reduces overfitting
- **More data** reduces overfitting

None of these provide *guarantees* (unlike differential privacy) but they are free mitigations that you're probably already applying for accuracy reasons.

In [ ]:
# Train a well-regularized model and compare vulnerability
print("Training regularized model (20 epochs, dropout=0.5, weight decay)...")
model_reg = MnistCNN(dropout=0.5).to(device)
opt_reg = torch.optim.Adam(model_reg.parameters(), lr=1e-3, weight_decay=1e-4)
for _ in range(20):
    model_reg.train()
    for x, y in train_loader:
        x, y = x.to(device), y.to(device)
        opt_reg.zero_grad(); F.cross_entropy(model_reg(x), y).backward(); opt_reg.step()

member_confs_reg,    _, _ = get_confidence_scores(model_reg, train_loader,     device)
nonmember_confs_reg, _, _ = get_confidence_scores(model_reg, nonmember_loader, device)

_, _, _, auc_reg = threshold_attack_roc(member_confs_reg, nonmember_confs_reg)

correct = total = 0
model_reg.eval()
with torch.no_grad():
    for x, y in train_loader:
        x, y = x.to(device), y.to(device)
        correct += (model_reg(x).argmax(1) == y).sum().item(); total += y.size(0)
train_acc_reg = correct / total

correct = total = 0
with torch.no_grad():
    for x, y in test_loader:
        x, y = x.to(device), y.to(device)
        correct += (model_reg(x).argmax(1) == y).sum().item(); total += y.size(0)
test_acc_reg = correct / total

print()
print("Model comparison:")
print(f"{'':20} {'Train acc':>12} {'Test acc':>10} {'Gen gap':>10} {'MI AUC':>10}")
print("-" * 68)
print(f"{'Overfit (no reg)':20} {train_acc:>12.4f} {test_acc:>10.4f} {train_acc-test_acc:>+10.4f} {auc:>10.4f}")
print(f"{'Regularized':20} {train_acc_reg:>12.4f} {test_acc_reg:>10.4f} {train_acc_reg-test_acc_reg:>+10.4f} {auc_reg:>10.4f}")
print()
print("Smaller generalization gap → lower MI AUC → less privacy leakage")

## 5. `memory-leak`: Reusable Membership Inference Auditor

The `memory-leak` tool from the project backlog is a CLI + library for auditing any model before deployment. The core function: run the threshold attack (cheap, fast) and report vulnerability metrics that are interpretable to a non-specialist.

Key outputs:
- **MI AUC**: overall attack effectiveness (0.5 = no leak, 1.0 = full leak)
- **TPR at FPR=0.1**: operationally meaningful metric — what fraction of members can be identified with only 10% false accusations?
- **Confidence gap**: mean confidence difference between members and non-members
- **Generalization gap**: train - test accuracy (the root cause)

In [ ]:
def memory_leak_audit(
    model: nn.Module,
    train_loader: torch.utils.data.DataLoader,
    holdout_loader: torch.utils.data.DataLoader,
    test_loader: torch.utils.data.DataLoader,
    device: torch.device,
    model_name: str = "model",
) -> dict:
    """
    memory-leak: membership inference vulnerability audit.

    Runs a threshold-based membership inference attack and
    reports key vulnerability metrics.

    Args:
        train_loader:   loader for training data (known members)
        holdout_loader: loader for held-out data (known non-members, same distribution)
        test_loader:    separate test set for generalization gap

    Returns:
        audit_report dict with vulnerability metrics
    """
    model.eval()

    # Collect confidence scores
    member_confs, _, member_preds      = get_confidence_scores(model, train_loader,   device)
    nonmember_confs, _, nonmember_preds = get_confidence_scores(model, holdout_loader, device)

    # Compute accuracies
    train_acc = np.mean(member_preds == np.concatenate([y for _, y in train_loader])[:len(member_preds)])
    # Use a simpler computation
    correct_train = correct_test = total_train = total_test = 0
    with torch.no_grad():
        for x, y in train_loader:
            x, y = x.to(device), y.to(device)
            correct_train += (model(x).argmax(1) == y).sum().item()
            total_train += y.size(0)
        for x, y in test_loader:
            x, y = x.to(device), y.to(device)
            correct_test += (model(x).argmax(1) == y).sum().item()
            total_test += y.size(0)
    train_acc = correct_train / total_train
    test_acc  = correct_test  / total_test

    # Membership inference metrics
    fpr_curve, tpr_curve, _, mi_auc = threshold_attack_roc(member_confs, nonmember_confs)
    fpr_10 = 0.1
    valid = fpr_curve <= fpr_10
    tpr_at_low_fpr = tpr_curve[valid].max() if valid.any() else 0.0

    report = {
        'model_name': model_name,
        'train_accuracy': train_acc,
        'test_accuracy': test_acc,
        'generalization_gap': train_acc - test_acc,
        'mean_confidence_members': float(member_confs.mean()),
        'mean_confidence_nonmembers': float(nonmember_confs.mean()),
        'confidence_gap': float(member_confs.mean() - nonmember_confs.mean()),
        'mi_auc': mi_auc,
        'tpr_at_fpr_01': tpr_at_low_fpr,
    }

    # Risk assessment
    if mi_auc > 0.75:
        risk = "HIGH — significant membership leakage detected"
    elif mi_auc > 0.60:
        risk = "MEDIUM — moderate membership leakage"
    else:
        risk = "LOW — minimal membership leakage"

    print(f"\nmemory-leak audit: {model_name}")
    print("=" * 55)
    print(f"  Train accuracy:              {train_acc:.4f}")
    print(f"  Test accuracy:               {test_acc:.4f}")
    print(f"  Generalization gap:          {train_acc - test_acc:.4f}")
    print(f"  Mean confidence (members):   {report['mean_confidence_members']:.4f}")
    print(f"  Mean confidence (non-mbrs):  {report['mean_confidence_nonmembers']:.4f}")
    print(f"  Confidence gap:              {report['confidence_gap']:.4f}")
    print(f"  Membership inference AUC:    {mi_auc:.4f}")
    print(f"  TPR @ FPR=0.1:               {tpr_at_low_fpr:.4f}")
    print(f"  Risk level: {risk}")
    return report


print("Auditing overfit model:")
report_overfit = memory_leak_audit(model_overfit, train_loader, nonmember_loader, test_loader, device, "Overfit CNN")

print("\nAuditing regularized model:")
report_reg = memory_leak_audit(model_reg, train_loader, nonmember_loader, test_loader, device, "Regularized CNN")

## Summary

```
Attack            Requires               AUC quality   Computational cost
──────────────────────────────────────────────────────────────────────────
Threshold         1 confidence score     Good           Negligible
Shadow model      k shadow models        Better         k × training cost
LiRA              2k models per record   Best           2k × training cost × n_records
```

**Key facts:**
- The generalization gap is the root cause — reduce it and you reduce vulnerability
- Confidence scores are the primary leakage channel — masking them reduces (but doesn't eliminate) attack effectiveness
- LiRA is the correct statistical framing: likelihood ratio test, not threshold on raw confidence
- Differential privacy provides the only formal guarantee (notebook 13)
- The `memory-leak` audit pattern: threshold attack + generalization gap + confidence gap → risk level

**Defenses by cost:**
1. **Regularization** (free): dropout, weight decay, early stopping — reduce overfitting, reduce leakage
2. **Confidence masking** (cheap): return top-k predictions without confidence scores
3. **DP-SGD** (expensive): provable ε-δ privacy guarantee at the cost of accuracy (notebook 14)

**Next:** Notebook 06 covers model inversion — the inverse attack. Where membership inference asks "was this record used for training?", model inversion asks "what does a representative training example look like?"